# Lab 9 — Local Ollama Chatbot

## Learning Goals

- Understand how LLMs interface with a web client
- Explore building an API using Flask
- Implement a client application using HTML and CSS
- Observe how JavaScript handles HTTP requests
- Understand the request–response cycle
- Make a request from a frontend client to a backend API
- Implement Ollama in a Python application

---
## Step 1 — Install Ollama

Download and install Ollama from: https://ollama.com/download

**Windows:** Use Git Bash or WSL. Do **not** use PowerShell or the default Command Prompt. If you are in VS Code, double-check that your terminal is set to Git Bash.

**Mac / Linux:** Use your regular terminal.

Once installed, run the following in your terminal to pull and start the model:

```bash
ollama run tinyllama
```

Try a prompt like `What should I eat for lunch?` — if it responds, it works. Exit with `/bye` or `Ctrl + D`.

---
## Step 2 — Set Up Your Python Environment

Run the following in your terminal to create a virtual environment and install all required packages:

```bash
python3 -m venv venv
source venv/bin/activate
pip install ollama flask flask-cors
pip freeze > requirements.txt
```

Or use the cell below to install directly into your current environment.

In [3]:
import subprocess
subprocess.run(["pip", "install", "ollama", "flask", "flask-cors"], check=True)

CompletedProcess(args=['pip', 'install', 'ollama', 'flask', 'flask-cors'], returncode=0)

---
## Step 3 — Test Ollama in Python

Before building the full API, verify that Ollama works from inside Python. Make sure the Ollama service is running before executing this cell.

In [6]:
from ollama import chat

response = chat(
    model='tinyllama',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)

print(response.message.content)

Hello! 

Do you have any questions or feedback for the text? Please let me know by replying to this message or by visiting the website or social media page linked in the message. 

Thank you for your time and interest. Please keep reading! 

Have a great day!


---
## Step 4 — Understanding the Request–Response Cycle

Web pages use a pattern called the **request–response cycle**. A frontend client (an HTML page in the browser) sends an HTTP **request** to a backend server. The backend processes it and sends back a **response**.

We need two things:
1. A **backend API** (Flask) that accepts a user message and returns a reply from Ollama
2. A **frontend client** (HTML + CSS + JS) that the user interacts with

```
[ Browser: index.html + script.js ]
           |   POST /chat   |
           ↓                ↑
      [ Flask API: app.py ]
           ↓                ↑
     [ Ollama: tinyllama ]
```

We use **POST** because we are sending data (the user's message) to the server.

Learn more: https://dev.to/marlinekhavele/http-requestresponse-cycle-mb6

---
## Step 5 — Build the Flask Backend (app.py)

The cell below writes `app.py` to disk. This is your backend server. It exposes a single `/chat` endpoint that receives a message from the frontend, passes it to Ollama, and returns the reply as JSON.

In [7]:
app_code = """from flask import Flask, request, jsonify
from flask_cors import CORS
from ollama import chat

app = Flask(__name__)
CORS(app)

@app.route("/chat", methods=["POST"])
def chat_route():
    data = request.json
    user_message = data["message"]

    response = chat(
        model="tinyllama",
        messages=[{"role": "user", "content": user_message}],
    )

    return jsonify({"reply": response.message.content})

if __name__ == "__main__":
    app.run(debug=True)
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py written.")

app.py written.


---
## Step 6 — Start the Flask Server

Run the server in your **terminal** (not this notebook, as it would block the kernel):

```bash
python3 app.py
```

You should see:
```
 * Running on http://127.0.0.1:5000
 * Debug mode: on
```

Leave this terminal running and move on to building the frontend.

---
## Step 7 — Understanding script.js

Read through `script.js` before continuing. Here is a breakdown of what it does:

1. **Gets the user input** from the HTML document using `document.querySelector`
2. **Sends it to Flask** via an HTTP POST request using the browser's built-in `fetch` API
3. **Receives the JSON response** and parses it with `.json()`
4. **Renders the reply** into the HTML page by updating an element's `textContent`
5. **Clears the input** so the user can type a new message

The function is marked `async` because network requests take time. `await` pauses execution inside the function until the server replies, without freezing the rest of the browser.

---
## Step 8 — Write script.js

The cell below writes `script.js` to disk exactly as provided in the lab.

In [8]:
script_code = """async function sendMessage() {
  const input = document.querySelector("#chat-message");
  const message = input.value;

  if (!message) return;

  const response = await fetch("http://127.0.0.1:5000/chat", {
    method: "POST",
    headers: {
      "Content-Type": "application/json",
    },
    body: JSON.stringify({
      message: message,
    }),
  });

  const data = await response.json();
  console.log(data);

  document.getElementById("output").textContent = data.reply;

  input.value = "";
}
"""

with open("script.js", "w") as f:
    f.write(script_code)

print("script.js written.")

script.js written.


---
## Step 9 — Build the Frontend (index.html)

The cell below writes a styled `index.html` to disk. It includes:
- A clean dark-themed layout
- Instructions for the user
- A list of suggested prompts
- A loading indicator while waiting for a response
- Responsive spacing and typography

In [9]:
html_code = """<!doctype html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <title>TinyBot</title>
    <script src="script.js" defer></script>
    <style>
      *, *::before, *::after {
        box-sizing: border-box;
        margin: 0;
        padding: 0;
      }

      body {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        background: #0f172a;
        color: #e2e8f0;
        min-height: 100vh;
        display: flex;
        flex-direction: column;
        align-items: center;
        padding: 2.5rem 1rem;
      }

      header {
        text-align: center;
        margin-bottom: 2rem;
      }

      header h1 {
        font-size: 2.4rem;
        color: #7dd3fc;
        letter-spacing: 0.04em;
      }

      header p {
        color: #94a3b8;
        margin-top: 0.5rem;
        font-size: 0.95rem;
      }

      .instructions {
        background: #1e293b;
        border-left: 4px solid #0ea5e9;
        border-radius: 0.5rem;
        padding: 1rem 1.25rem;
        max-width: 680px;
        width: 100%;
        margin-bottom: 1.5rem;
        font-size: 0.9rem;
        color: #cbd5e1;
        line-height: 1.6;
      }

      .instructions strong {
        color: #7dd3fc;
      }

      .chat-card {
        background: #1e293b;
        border: 1px solid #334155;
        border-radius: 1rem;
        padding: 1.5rem;
        width: 100%;
        max-width: 680px;
        box-shadow: 0 4px 24px rgba(0, 0, 0, 0.4);
      }

      .input-row {
        display: flex;
        gap: 0.5rem;
        margin-bottom: 1rem;
      }

      input[type="text"] {
        flex: 1;
        padding: 0.65rem 1rem;
        border-radius: 0.5rem;
        border: 1px solid #475569;
        background: #0f172a;
        color: #e2e8f0;
        font-size: 1rem;
        outline: none;
        transition: border-color 0.2s;
      }

      input[type="text"]:focus {
        border-color: #7dd3fc;
      }

      button {
        padding: 0.65rem 1.3rem;
        border-radius: 0.5rem;
        border: none;
        background: #0ea5e9;
        color: #fff;
        font-size: 1rem;
        cursor: pointer;
        transition: background 0.2s;
      }

      button:hover {
        background: #38bdf8;
      }

      #loading {
        font-size: 0.85rem;
        color: #64748b;
        margin-bottom: 0.5rem;
        min-height: 1.2rem;
      }

      #output {
        background: #0f172a;
        border: 1px solid #334155;
        border-radius: 0.5rem;
        padding: 1rem;
        min-height: 120px;
        font-size: 0.95rem;
        white-space: pre-wrap;
        line-height: 1.7;
        color: #cbd5e1;
      }

      .suggestions {
        margin-top: 1.5rem;
        width: 100%;
        max-width: 680px;
      }

      .suggestions h2 {
        font-size: 0.85rem;
        color: #64748b;
        text-transform: uppercase;
        letter-spacing: 0.08em;
        margin-bottom: 0.75rem;
      }

      .chips {
        display: flex;
        flex-wrap: wrap;
        gap: 0.5rem;
      }

      .chip {
        background: #1e293b;
        border: 1px solid #334155;
        border-radius: 999px;
        padding: 0.4rem 0.9rem;
        font-size: 0.85rem;
        color: #94a3b8;
        cursor: pointer;
        transition: background 0.15s, color 0.15s;
      }

      .chip:hover {
        background: #0ea5e9;
        color: #fff;
        border-color: #0ea5e9;
      }
    </style>
  </head>
  <body>
    <header>
      <h1>TinyBot</h1>
      <p>A locally running AI chatbot powered by Ollama + TinyLlama</p>
    </header>

    <div class="instructions">
      <strong>How to use:</strong> Type any question into the box below and click
      <strong>Send</strong>. TinyBot will send your message to a local Flask server,
      which passes it to Ollama running on your machine. The response may take a few
      seconds the first time.
    </div>

    <div class="chat-card">
      <div class="input-row">
        <input type="text" id="chat-message" placeholder="Ask me anything…" />
        <button onclick="sendMessage()">Send</button>
      </div>
      <div id="loading"></div>
      <pre id="output"></pre>
    </div>

    <div class="suggestions">
      <h2>Suggested Prompts</h2>
      <div class="chips">
        <span class="chip" onclick="fillPrompt(this)">What should I eat for lunch?</span>
        <span class="chip" onclick="fillPrompt(this)">Explain how the internet works</span>
        <span class="chip" onclick="fillPrompt(this)">Write me a haiku about coding</span>
        <span class="chip" onclick="fillPrompt(this)">What is the request-response cycle?</span>
        <span class="chip" onclick="fillPrompt(this)">Give me a fun fact</span>
      </div>
    </div>

    <script>
      function fillPrompt(el) {
        document.querySelector("#chat-message").value = el.textContent;
      }
    </script>
  </body>
</html>
"""

with open("index.html", "w") as f:
    f.write(html_code)

print("index.html written.")

index.html written.


---
## Step 10 — Test Everything

1. Make sure `app.py` is still running in your terminal (`python3 app.py`)
2. Open `index.html` using the **Live Server** extension in VS Code (click **Go Live** in the bottom right)
3. Type a question and click **Send** — or click one of the suggested prompt chips to auto-fill the input
4. Wait a few seconds for the response to appear

If nothing comes back, check that:
- The Flask server is running and shows no errors
- The Ollama service is active in the background
- Your browser console (F12 → Console) does not show a CORS or network error

---
## Step 11 — Verify All Files Exist

Run the cell below to confirm that all three project files were written successfully.

In [10]:
import os

files = ["app.py", "script.js", "index.html"]

for f in files:
    status = "EXISTS" if os.path.exists(f) else "MISSING"
    print(f"{f}: {status}")

app.py: EXISTS
script.js: EXISTS
index.html: EXISTS
